# 工具的应用案例

In [14]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model= "deepseek-v4-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

from pydantic import BaseModel,Field

class WeatherSchema(BaseModel):
    city: str = Field(default="上海", description="具体的城市名称")

# 定义工具
@tool("get_weather_and_forecast", description='查询当日的天气，可以包含明天的天气')
def get_weather(city: str,if_forecast:bool):
    res = f"{city}的今日天气是下雨"
    if if_forecast:
        res += f", {city} 的明日天气是晴天"
    return res

print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日的天气，可以包含明天的天气', 'parameters': {'properties': {'city': {'type': 'string'}, 'if_forecast': {'type': 'boolean'}}, 'required': ['city', 'if_forecast'], 'type': 'object'}}}


In [15]:
from langchain_core.messages import HumanMessage
# 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 维护一个消息列表
messages = [HumanMessage("今天杭州的天气怎么样？明天呢？")]

# 调用模型
response = model_with_tools.invoke(messages)

messages.append(response)

# 获取响应的tool_call字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather_and_forecast':
        # 调用工具，因为大模型不能直接调用工具，所以我们主动让工具调用执行
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 调用ai模型，得到AIMessages
final_response = model.invoke(messages)

messages.append(final_response)

for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_QfcVKGSf3ScRiUS7Kk619357)
 Call ID: call_00_QfcVKGSf3ScRiUS7Kk619357
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州的今日天气是下雨, 杭州 的明日天气是晴天
================================== Ai Message ==================================

今天杭州的天气是**下雨**，明天是**晴天**。
